In [ ]:
# 라이브러리 설치 (호환성 문제 해결을 위해 kaleido 버전을 0.2.1로 고정)
!pip install pykrx openai plotly==5.24.1 kaleido==0.2.1 -q
!pip install --upgrade numpy -q
!pip install --upgrade pandas -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which

In [ ]:
import base64
from datetime import datetime, timedelta

import pandas as pd
import plotly.graph_objects as go
from pykrx import stock
from openai import OpenAI

KRX 로그인 실패: KRX_ID 또는 KRX_PW 환경 변수가 설정되지 않았습니다.


In [ ]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# 키 점검 (앞 7자리만 찍어보기)
print(f"Key prefix: {OPENAI_API_KEY[:7]}... (length: {len(OPENAI_API_KEY)})")

client = OpenAI(api_key=OPENAI_API_KEY)

Key prefix: sk-proj... (length: 164)


In [ ]:
target_stocks = {
    "삼성전자": "005930",
    "SK하이닉스": "000660",
    "한미반도체": "042700",
}

# 오늘 기준 1년치
end = datetime.today().strftime("%Y%m%d")
start = (datetime.today() - timedelta(days=365)).strftime("%Y%m%d")
print(f"조회 기간: {start} ~ {end}")

조회 기간: 20250416 ~ 20260416


In [ ]:
dfs = {}
for name, ticker in target_stocks.items():
    df = stock.get_market_ohlcv_by_date(start, end, ticker)
    dfs[name] = df
    print(f"✅ {name} ({ticker}): {len(df)}일치 수집")

# 첫 종목 데이터 미리보기
first_name = list(dfs.keys())[0]
dfs[first_name].tail()

✅ 삼성전자 (005930): 244일치 수집
✅ SK하이닉스 (000660): 244일치 수집
✅ 한미반도체 (042700): 244일치 수집


,시가,고가,저가,종가,거래량,등락률
날짜,,,,,,
2026-04-10,208500,211000,205500,206000,18244459,0.980392
2026-04-13,198200,203000,198200,201000,19603415,-2.427184
2026-04-14,208000,210000,205500,206500,23672078,2.736318
2026-04-15,215000,215500,210000,211000,24092884,2.179177
2026-04-16,212000,218000,210500,217500,21339292,3.080569


In [ ]:
# 모든 종목에 MA 컬럼 추가
for name, df in dfs.items():
    df['MA5']  = df['종가'].rolling(window=5).mean()
    df['MA20'] = df['종가'].rolling(window=20).mean()
    df['MA60'] = df['종가'].rolling(window=60).mean()

dfs[first_name][['종가', 'MA5', 'MA20', 'MA60']].tail()

,종가,MA5,MA20,MA60
날짜,,,,
2026-04-10,206000,202020.0,191180.0,178060.000000
2026-04-13,201000,203600.0,191795.0,179116.666667
2026-04-14,206500,205600.0,192425.0,180220.000000
2026-04-15,211000,205700.0,192550.0,181338.333333
2026-04-16,217500,208400.0,193400.0,182481.666667


In [ ]:
def make_chart(name: str, df: pd.DataFrame) -> go.Figure:
    """한 종목의 캔들차트 + 이평선 3개를 그린다."""
    fig = go.Figure(data=[
        go.Candlestick(
            x=df.index,
            open=df['시가'], high=df['고가'],
            low=df['저가'], close=df['종가'],
            name='캔들',
            increasing_line_color='#EF4444',  # 상승 빨강 (한국 관례)
            decreasing_line_color='#3B82F6',  # 하락 파랑
        ),
        go.Scatter(x=df.index, y=df['MA5'],  line=dict(color='orange',    width=1.3), name='MA5'),
        go.Scatter(x=df.index, y=df['MA20'], line=dict(color='royalblue', width=1.3), name='MA20'),
        go.Scatter(x=df.index, y=df['MA60'], line=dict(color='firebrick', width=1.3), name='MA60'),
    ])
    fig.update_layout(
        title=f'{name} — 1년 주가 & 이동평균선',
        template='plotly_dark',
        xaxis_rangeslider_visible=False,
        height=600,
        legend=dict(orientation='h', y=1.05),
    )
    return fig

# 첫 종목 시각화
fig = make_chart(first_name, dfs[first_name])
fig.show()

In [ ]:
def encode_image(path: str) -> str:
    """이미지 파일을 base64 문자열로 인코딩."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [ ]:
SYSTEM_PROMPT = """
너는 월스트리트 15년차 헤지펀드 퀀트 애널리스트야.
첨부된 주가 차트의 이동평균선(주황=MA5, 파랑=MA20, 빨강=MA60) 배열과
캔들 패턴을 분석해. 금융 전문 용어를 쓰되 초보자도 이해할 수 있게 설명해.

반드시 아래 형식으로만 답해:

📊 현재 추세: (한 줄 요약)
🎯 핵심 시그널: (골든크로스/데드크로스/정배열/역배열 등 구체적 패턴 1~2개)
💡 투자 의견: [매수 / 보유 / 매도] 중 하나 + 3줄 이내 근거
⚠️ 리스크: (놓치면 안 될 하방 리스크 한 줄)
"""

def analyze_with_ai(name: str, image_path: str) -> str:
    """GPT-4o Vision에게 차트 이미지를 보내서 분석 텍스트를 받는다."""
    image_b64 = encode_image(image_path)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "text",
                 "text": f"이 차트는 {name}의 최근 1년 일봉이야. 이평선 배열을 중심으로 분석해줘."},
                {"type": "image_url",
                 "image_url": {"url": f"data:image/png;base64,{image_b64}"}}
            ]}
        ],
        max_tokens=500,
        temperature=0.3,
    )
    return response.choices[0].message.content

In [ ]:
# 첫 종목으로 테스트
# engine='kaleido'를 명시하여 이미지를 저장합니다.
fig.write_image("chart.png", width=1200, height=700, scale=2, engine="kaleido")
analysis = analyze_with_ai(first_name, "chart.png")
print(f"🤖 AI 퀀트 의견 — {first_name}\n{'='*40}")
print(analysis)

🤖 AI 퀀트 의견 — 삼성전자
📊 현재 추세: 상승 추세

🎯 핵심 시그널: 정배열 (MA5 > MA20 > MA60)

💡 투자 의견: 매수
- 이동평균선이 정배열을 이루며 상승세를 보이고 있습니다.
- 최근 조정 후 다시 상승하는 모습이 나타나고 있습니다.
- 거래량도 증가하여 추가 상승 가능성이 높습니다.

⚠️ 리스크: 단기 과매수로 인한 조정 가능성


In [ ]:
results = {}

for name, df in dfs.items():
    print(f"\n⏳ {name} 분석 중...")
    fig = make_chart(name, df)
    img_path = f"chart_{name}.png"
    fig.write_image(img_path, width=1200, height=700, scale=2)
    analysis = analyze_with_ai(name, img_path)
    results[name] = analysis
    print(f"✅ {name} 완료")

# 결과 출력
for name, text in results.items():
    print(f"\n{'='*50}\n🤖 {name}\n{'='*50}\n{text}")


⏳ 삼성전자 분석 중...
✅ 삼성전자 완료

⏳ SK하이닉스 분석 중...
✅ SK하이닉스 완료

⏳ 한미반도체 분석 중...
✅ 한미반도체 완료

🤖 삼성전자
📊 현재 추세: 상승 추세

🎯 핵심 시그널: 정배열 (MA5 > MA20 > MA60)

💡 투자 의견: 매수
- 이동평균선이 정배열을 유지하며 상승세를 보이고 있습니다.
- 최근 조정 후 다시 상승세로 전환되는 모습입니다.
- 거래량 증가가 상승 추세를 지지하고 있습니다.

⚠️ 리스크: 단기 과매수로 인한 조정 가능성

🤖 SK하이닉스
📊 현재 추세: 상승 추세

🎯 핵심 시그널: 정배열 (MA5 > MA20 > MA60)

💡 투자 의견: 매수
- 이동평균선이 정배열을 이루며 상승세를 보이고 있습니다.
- 최근 거래량 증가와 함께 주가가 상승하고 있어 추가 상승 가능성이 높습니다.
- 단기 조정이 있을 수 있으나 전반적인 상승 추세가 유지되고 있습니다.

⚠️ 리스크: 단기 과매수로 인한 조정 가능성

🤖 한미반도체
📊 현재 추세: 상승 추세

🎯 핵심 시그널: 정배열 (MA5 > MA20 > MA60), 최근 MA5와 MA20의 골든크로스 발생

💡 투자 의견: 매수
- 최근 골든크로스가 발생하며 상승 모멘텀이 강화되고 있습니다.
- MA60이 상승세를 유지하며 장기적인 상승 추세를 지지하고 있습니다.
- 단기 조정 후 반등이 예상됩니다.

⚠️ 리스크: 단기 변동성 증가 가능성
